In [ ]:
import os
import re
import pandas as pd
import sys
import matplotlib.pyplot as plt  # For plotting


In [ ]:
extracted_folder = './data/pb/'  # Folder where the extracted files will be stored
csv_output_folder = extracted_folder+'csv_samples'
plot_output_folder = extracted_folder+'plots'  # Optional: if you want to save plots as images


In [25]:
os.makedirs(csv_output_folder, exist_ok=True)
os.makedirs(plot_output_folder, exist_ok=True)

In [ ]:
for root, dirs, files in os.walk(extracted_folder):
    for file in files:
        if file.lower().endswith('.swp'):
            swp_file_path = os.path.join(root, file)
            
            # --- Extract label from filename ---
            # Example: "Pb 0.01 L1.swp" → label: 0.01
            label_match = re.search(r'Pb\s*([\d\.]+)', file)
            if label_match:
                label = float(label_match.group(1))
            else:
                label = None  # or skip if label is essential
            
            # --- Read and parse numeric data ---
            lines_of_data = []
            with open(swp_file_path, 'r') as f:
                for line in f:
                    line = line.strip()
                    # Skip empty lines
                    if not line:
                        continue
                    
                    parts = line.split()
                    # Keep lines that have exactly two tokens, both numeric
                    if len(parts) == 2:
                        try:
                            float(parts[0])
                            float(parts[1])
                            lines_of_data.append(parts)
                        except ValueError:
                            pass
            
            # If no numeric data found, skip file
            if not lines_of_data:
                print(f"No numeric data found in {file}. Skipping...")
                continue
            
            # --- Create a DataFrame with columns U(V) and j(mA) ---
            df = pd.DataFrame(lines_of_data, columns=['U(V)', 'j(mA)'])
            # Convert columns to float
            df = df.astype(float)
            
            # Add the label as a column
            # df['label'] = label
            
            # --- Save to CSV ---
            csv_file_name = os.path.splitext(file)[0] + '.csv'
            csv_file_path = os.path.join(csv_output_folder, csv_file_name)
            df.to_csv(csv_file_path, index=False)
            print(f"Processed '{file}' → '{csv_file_path}'")
            
            # -----------------------------
            # 4. Plot U(V) vs j(mA)
            # -----------------------------
            plt.figure(figsize=(6, 4))
            plt.plot(df['U(V)'].to_numpy(), df['j(mA)'].to_numpy(), marker='o', linestyle='-')
            plt.xlabel("U(V)")
            plt.ylabel("j(mA)")
            plot_title = f"{file} (Label: {label})"
            plt.title(plot_title)
            plt.tight_layout()
            
            # Show or Save the plot
            # Option 1: Show the plot (each file will open a figure)
            # plt.show()
            
            # Option 2: Save the plot to disk
            plot_png_name = os.path.splitext(file)[0] + '.png'
            plot_png_path = os.path.join(plot_output_folder, plot_png_name)
            plt.savefig(plot_png_path)
            plt.close()  # Close the figure to avoid memory issues if many files